In [ ]:
# pip install dspy-ai
import dspy

# =========================================================
# 1. Khởi tạo LLM nội bộ (Ollama chạy ở localhost:11434)
# =========================================================
llm = dspy.OllamaLocal(
    model="llama3.2:3b",  # tên model trong Ollama (không cần prefix ollama/)
    base_url="http://localhost:11434",
    timeout_s=300
)

# Đăng ký LLM cho toàn bộ DSPy runtime
dspy.configure(lm=llm)

# =========================================================
# 2. Định nghĩa Signature (task contract)
# =========================================================
class SlideGen(dspy.Signature):
    """
    Document -> Slide (JSON)
    """
    document: str = dspy.InputField(desc="Input document text to convert into a slide")
    style: str = dspy.InputField(desc="Style guide for the slide generation")

    slide_json: dict = dspy.OutputField(
        desc="""
        Valid JSON only.
        Fields:
          - title: short slide title (<= 8 words)
          - bullets: 3–5 concise bullet points
        No extra text, no markdown.
        """
    )

# =========================================================
# 3. Định nghĩa DSPy Module
# =========================================================
class SlideGenerator(dspy.Module):
    def forward(self, document, style):
        return dspy.Predict(SlideGen)(
            document=document,
            style=style
        )

# =========================================================
# 4. Chạy thử
# =========================================================
if __name__ == "__main__":
    generator = SlideGenerator()

    doc = """
    Deep learning models require large datasets to generalize well.
    Overfitting occurs when models are too complex relative to the data.
    Regularization methods such as dropout help reduce overfitting.
    """

    result = generator(
        document=doc,
        style="gentle, academic, slide-friendly"
    )

    print(result.slide_json)


In [6]:
import dspy
rm = dspy.Retrieve(k=5)
data_list = rm(query_or_queries=["Park Jihyo là ai"])

AssertionError: No RM is loaded.

In [7]:
print("hehe")

hehe


In [12]:
text = """
question,label\n"Nguyễn Minh Phương là ai?",1 "Số điện thoại của Nguyễn Tuấn Đức là gì?",1 "annk3 là ai?",1 "Email của Nguyễn Văn A là gì?",1 "annk3 làm việc ở bộ phận nào?",1 "Năm sinh của hungnq71 là bao nhiêu?",1 "Bao nhiêu nhân sự thuộc khối CNTT?",1 "Tỷ lệ nghỉ việc năm 2024 là bao nhiêu?",1 "Danh sách nhân sự có KPI xuất sắc quý này?",1 "Số lao động hiện có của công ty trong tháng 1 năm 2025 là bao nhiêu?",1 "Nhu cầu nhân sự của công ty trong quý 2 năm 2025 là bao nhiêu?",1 "Số lao động tăng mới trong tháng 6 năm 2025 là bao nhiêu?",1 "Hiện tại số lượng nhân sự trong danh sách và hợp đồng dịch vụ là bao nhiêu?",1 "Số lượng nhân sự theo từng level hiện nay?",1 "Số lượng nhân sự theo thâm niên trên 5 năm là bao nhiêu người?",1 "So sánh hiệu suất làm việc giữa nhân sự Junior và Senior.",1
"""

In [13]:
import pandas as pd
from io import StringIO
df = pd.read_csv(StringIO(text))
df

,,,,,,,,,,,,,,,question,label
Nguyễn Minh Phương là ai?,"1 ""Số điện thoại của Nguyễn Tuấn Đức là gì?""","1 ""annk3 là ai?""","1 ""Email của Nguyễn Văn A là gì?""","1 ""annk3 làm việc ở bộ phận nào?""","1 ""Năm sinh của hungnq71 là bao nhiêu?""","1 ""Bao nhiêu nhân sự thuộc khối CNTT?""","1 ""Tỷ lệ nghỉ việc năm 2024 là bao nhiêu?""","1 ""Danh sách nhân sự có KPI xuất sắc quý này?""","1 ""Số lao động hiện có của công ty trong tháng 1 năm 2025 là bao nhiêu?""","1 ""Nhu cầu nhân sự của công ty trong quý 2 năm 2025 là bao nhiêu?""","1 ""Số lao động tăng mới trong tháng 6 năm 2025 là bao nhiêu?""","1 ""Hiện tại số lượng nhân sự trong danh sách và hợp đồng dịch vụ là bao nhiêu?""","1 ""Số lượng nhân sự theo từng level hiện nay?""","1 ""Số lượng nhân sự theo thâm niên trên 5 năm là bao nhiêu người?""","1 ""So sánh hiệu suất làm việc giữa nhân sự Jun...",1


In [18]:
import pandas as pd
import os

excel_path = r"C:\Users\nka15\OneDrive\Máy tính\hehe\data.xlsx"
output_dir = r"C:\Users\nka15\OneDrive\Máy tính\hehe"

# Đọc tất cả sheet
sheets = pd.read_excel(excel_path, sheet_name=None)

for sheet_name, df in sheets.items():
    # Đảm bảo tất cả dữ liệu là string để tránh lỗi so sánh str/int
    df = df.astype(str)

    # Xuất ra CSV
    output_path = os.path.join(output_dir, f"{sheet_name}.csv")
    df.to_csv(output_path, index=False, encoding="utf-8-sig")

print("Đã xuất tất cả sheet thành file CSV riêng.")


Đã xuất tất cả sheet thành file CSV riêng.


In [51]:
import re
from collections import OrderedDict

def outline_md_to_number(outline_md: str) -> tuple[dict, str]:
        """
        Convert a markdown outline (#, ##, ###, ...) into a numbered hierarchical dict
        and a numbered markdown string.
        Leaf nodes are marked with -1.

        INPUT
        # Section 1 name
        ## subsection 1 name
        ### subsubsection 1 name
        ### subsubsection 2 name
        ## subsection 2 name
        ### subsubsection 1 name
        ## subsection 3 name
        # Section 2 name
        # Section 3 name
        OUTPUT:
        output_dict = {
            "1. Section 1 name": {
                "1.1. subsection 1 name": {
                    "1.1.1. subsubsection 1 name": -1,
                    "1.1.2. subsubsection 2 name": -1
                },
                "1.2. subsection 2 name": {
                    "1.2.1. subsubsection 1 name": -1
                },
                "1.3. subsection 3 name": -1
            },
            "2. Section 2 name": -1,
            "3. Section 3 name": -1
        }
        output_md = 
        1. Section 1 name
        1.1. subsection 1 name
        1.1.1. subsubsection 1 name
        1.1.2. subsubsection 2 name
        1.2. subsection 2 name
        1.2.1. subsubsection 1 name
        1.3. subsection 3 name
        2. Section 2 name
        3. Section 3 name
        """
        lines = [line.rstrip() for line in outline_md.splitlines() if line.strip()]
        pattern = re.compile(r'^(#+)\s+(.*)$')

        # Stack holds tuples: (level, dict_ref, index_path)
        # index_path = [1, 2, 1]  -> "1.2.1."
        stack = []

        root = OrderedDict()
        counters = []
        numbered_lines = []

        for line in lines:
            match = pattern.match(line)
            if not match:
                continue

            level = len(match.group(1))
            title = match.group(2).strip()

            # Adjust counters depth
            while len(counters) < level:
                counters.append(0)
            while len(counters) > level:
                counters.pop()

            counters[-1] += 1
            counters[level - 1 + 1:] = []

            number = ".".join(str(c) for c in counters) + "."
            key = f"{number} {title}"
            
            # Build numbered markdown line (without # prefix)
            numbered_lines.append(f"{number} {title}")

            # Pop stack until parent level
            while stack and stack[-1][0] >= level:
                stack.pop()

            if not stack:
                root[key] = -1
                stack.append((level, root, key))
            else:
                parent_dict = stack[-1][1][stack[-1][2]]
                if parent_dict == -1:
                    parent_dict = OrderedDict()
                    stack[-1][1][stack[-1][2]] = parent_dict

                parent_dict[key] = -1
                stack.append((level, parent_dict, key))
        
        numbered_md = "\n".join(numbered_lines)
        return root, numbered_md


md = """
# Section 1 name
## subsection 1 name
### subsubsection 1 name
#### hehehe
### subsubsection 2 name
#### shhehehc
##### ahadhaj
######jcsijc
## subsection 2 name
### subsubsection 1 name
## subsection 3 name
# Section 2 name
# Section 3 name
"""

import json
_, outline_numbered_md= outline_md_to_number(md)
outline_numbered_md


'1. Section 1 name\n1.1. subsection 1 name\n1.1.1. subsubsection 1 name\n1.1.1.1. hehehe\n1.1.2. subsubsection 2 name\n1.1.2.1. shhehehc\n1.1.2.1.1. ahadhaj\n1.2. subsection 2 name\n1.2.1. subsubsection 1 name\n1.3. subsection 3 name\n2. Section 2 name\n3. Section 3 name'

In [55]:
def get_relevant_context(section_key: str, outline_numbered_md: str):
        """
        Extract all section keys that appear before the given section_key,
        stopping when reaching a level-1 section.
        
        Args:
            section_key: The target section key (e.g., "1.1.2. subsubsection 2 name")
            outline_numbered_md: Numbered markdown outline string (without '#' symbols)
            slides_content: Dictionary of slide content
            
        Returns:
            List of section keys that appear before section_key, up to the first level-1 section
            
        Example:
            If section_key = "1.1.2. subsubsection 2 name", returns:
            ["1. Section 1 name", "1.1. subsection 1 name", "1.1.1. subsubsection 1 name"]
            
            If section_key = "2. Section 2 name", returns: []
        """
        lines = [line.rstrip() for line in outline_numbered_md.splitlines() if line.strip()]
        # Pattern to match numbered sections like "1. Title" or "1.1.2. Title"
        pattern = re.compile(r'^([\d.]+)\s+(.*)$')
        
        all_sections = []
        current_level1 = None
        sections_after_level1 = []
        
        for line in lines:
            match = pattern.match(line)
            if not match:
                continue
            
            # Count dots to determine level: "1." = 1 dot = level 1, "1.1." = 2 dots = level 2
            numbering = match.group(1)
            level = numbering.count('.')
            section_name = line.strip()  # Use the full line as section name
            
            # Check if this is our target section
            if section_name == section_key:
                # Found the target, combine level-1 section with all sections after it
                if current_level1:
                    all_sections = [current_level1] + sections_after_level1
                else:
                    all_sections = sections_after_level1
                break
            
            # Track sections
            if level == 1:
                # New level-1 section found, reset tracking
                current_level1 = section_name
                sections_after_level1 = []
            else:
                # This is a subsection, add it to the list
                sections_after_level1.append(section_name)
        
        relevant_sections = all_sections
        return relevant_sections
get_relevant_context("1.1.1.1. hehehe", outline_numbered_md)

['1. Section 1 name', '1.1. subsection 1 name', '1.1.1. subsubsection 1 name']

In [34]:
def get_level1_child(tree: dict, name: str) -> list:
    """
    Given a hierarchical outline dict and a node name,
    return a list of all level-1 children names of that node.

    Level-1 children = direct children only (not deeper descendants).

    If:
    - node does not exist
    - node is a leaf (-1)
    → return empty list
    """

    def dfs(current: dict) -> list | None:
        for k, v in current.items():
            if k == name:
                if v == -1:
                    return []
                return list(v.keys())
            if isinstance(v, dict):
                found = dfs(v)
                if found is not None:
                    return found
        return None

    result = dfs(tree)
    return result if result is not None else []

get_level1_child(tree, "1.1.2.1. shhehehc")

AttributeError: 'tuple' object has no attribute 'items'

TypeError: get_relevant_context() missing 1 required positional argument: 'outline_numbered_md'

In [ ]:


file_path = r"D:\python\LecSlideGen\data\raw\SinhHoc10_B17.pdf"


Started parsing the file under job_id cbaafeec-c80f-4200-be55-ec004a152ebf


In [7]:
documents

[Document(id_='86359f8b-8e5f-46d9-941f-bb8b03f18441', embedding=None, metadata={}, excluded_embed_metadata_keys=[], excluded_llm_metadata_keys=[], relationships={}, metadata_template='{key}: {value}', metadata_separator='\n', text_resource=MediaResource(embeddings=None, data=None, text='\n# BÀI 17 GIẢM PHÂN\n\n# YÊU CẦU CẦN ĐẠT\n\nDựa vào cơ chế nhân đôi và phân li của NST để giải thích được quá trình phân, giảm thụ tinh cùng với nguyên phân là cơ sở của sinh sản hữu tính &#x26; sinh vật.\n\nTrình bày được một số nhân tố ảnh hưởng đến quá trình giảm phân.\n\nLập được bảng so sánh quá trình nguyên phân và quá trình giảm phân.\n\nVận dụng kiến thức về nguyên phân và giảm phân vào giải thích một số vấn đề trong thực tiễn.\n\nCơ chế nào giúp các loài sinh sản hữu tính duy trì được bộ NST của loài qua các thế hệ?\n\n# DIỄN BIẾN CỦA GIẢM PHÂN\n\nGiảm phân (phân bào giảm nhiễm) là hình thức phân chia của các tế bào mầm sinh dục trong trình sản sinh giao tử ở các cơ quan sinh sản. Giảm phân gồ


# BÀI 17 GIẢM PHÂN

# YÊU CẦU CẦN ĐẠT

Dựa vào cơ chế nhân đôi và phân li của NST để giải thích được quá trình phân, giảm thụ tinh cùng với nguyên phân là cơ sở của sinh sản hữu tính &#x26; sinh vật.

Trình bày được một số nhân tố ảnh hưởng đến quá trình giảm phân.

Lập được bảng so sánh quá trình nguyên phân và quá trình giảm phân.

Vận dụng kiến thức về nguyên phân và giảm phân vào giải thích một số vấn đề trong thực tiễn.

Cơ chế nào giúp các loài sinh sản hữu tính duy trì được bộ NST của loài qua các thế hệ?

# DIỄN BIẾN CỦA GIẢM PHÂN

Giảm phân (phân bào giảm nhiễm) là hình thức phân chia của các tế bào mầm sinh dục trong trình sản sinh giao tử ở các cơ quan sinh sản. Giảm phân gồm hai lần phân bào liên tiếp là giảm phân I và giảm phân II. Trước khi tế bào bước vào giảm phân I, ở kỳ trung gian, mỗi NST được nhân đôi tạo thành NST kép.

# 1. Giảm phân

| Chromatid chị em   | Chromatid không chị em                     |
| ------------------ | ---------------------------------------

In [2]:
import os
from marker.converters.pdf import PdfConverter
from marker.models import create_model_dict
from marker.output import text_from_rendered
from marker.config.parser import ConfigParser

# ==============================
# CONFIG
# ==============================

PDF_PATH = r"D:\python\LecSlideGen\data\raw\SinhHoc10_B8.pdf"
OUTPUT_MD = r"D:\python\LecSlideGen\data\processed\SinhHoc10_B8.md"

config_dict = {
    "output_format": "markdown",
    "use_llm": True,

    # Ép dùng OpenAIService
    "llm_service": "marker.services.openai.OpenAIService",

    # Ollama config
    "openai_base_url": "http://112.137.129.245:33046/v1",
    "openai_api_key": "ollama",
    "openai_model": "hf.co/Qwen/Qwen3-8B-GGUF:Q8_0",
}

parser = ConfigParser(config_dict)

converter = PdfConverter(
    config=parser.generate_config_dict(),
    artifact_dict=create_model_dict(),
    processor_list=parser.get_processors(),
    renderer=parser.get_renderer(),
    llm_service=parser.get_llm_service(),
)

print("🔍 Converting PDF with Ollama LLM...")

rendered = converter(PDF_PATH)

text, markdown, images = text_from_rendered(rendered)

print(text)

🔍 Converting PDF with Ollama LLM...


Recognizing Text: 100%|██████████| 57/57 [24:34<00:00, 25.87s/it]  
Detecting bboxes: 0it [00:00, ?it/s]
LLM processors running: 0it [00:00, ?it/s]
Running LLMSectionHeaderProcessor:   0%|          | 0/1 [00:00<?, ?it/s]2026-02-23 11:58:44,664 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 3 seconds... (Attempt 1/3)
2026-02-23 11:59:52,000 [WARNING] marker: Rate limit error: Request timed out.. Retrying in 6 seconds... (Attempt 2/3)
2026-02-23 12:01:02,466 [ERROR] marker: Rate limit error: Request timed out.. Max retries reached. Giving up. (Attempt 3/3)
2026-02-23 12:01:02,467 [WARNING] marker: LLM did not return a valid response
Running LLMSectionHeaderProcessor: 100%|██████████| 1/1 [03:24<00:00, 204.34s/it]

Khác với tế bào nhân sơ, tế bào nhân thực có kích thước lớn và cấu tạo phức tạp hơn nhiều. Đó là: vật chất di truyền được bao bọc bởi lớp màng tạo nên cấu trúc gọi là nhân tế bào, bên trong tế bào chất các hệ thống màng chia tế bào thành các xoang riêng biệt. Ngoài ra, trong tế bào chất của tế bào nhân thực còn có nhiều bào quan có màng bao bọc.

![](_page_0_Picture_2.jpeg)

a) Tế bào động vật

![](_page_0_Picture_4.jpeg)

Hình 8.1. Cấu trúc tổng thể của tế bào nhân thực

b) Tế bào thực vật

### I – NHÂN TẾ BÀO

Nhân tế bào phần lớn có hình cầu với đường kính khoảng 5µm, được bao bọc bởi 2 lớp màng, bên trong là dịch nhân chứa chất nhiễm sắc (gồm ADN liên kết với prôtêin) và nhân con (hình 8.1). Trên màng nhân thường có nhiều lỗ nhỏ.

▼ Một nhà khoa học đã tiến hành phá huỷ nhân của tế bào trứng ếch thuộc loài A, sau đó lấy nhân của tế bào sinh dưỡng của loài B cấy vào. Sau nhiều lần thí nghiệm, ông đã nhận được các con ếch con từ các tế bào đã được chuyển nhân.

Em hãy cho biết các co